# 2. Brute-Force Baseline

Brute force is the simplest exact nearest-neighbour method: compute the distance from the query to **every** database vector and keep the *k* smallest.

This notebook:

1. implements brute-force k-NN search (k = 10),
2. uses it to compute the **ground truth**, the true 10 nearest neighbours of each of the 500 queries,
3. measures its **query time**, the baseline every other method is compared against.

Shared evaluation code (recall@k and timing) lives in `evaluation.py`.

In [1]:
# Check the working directory: the data paths below are relative to the project folder.
from pathlib import Path

print(Path.cwd())

/Users/katrinarnadottir/Downloads/ALG_Group_Project


## Load the data

Load the embeddings and sentences produced in `01_data_preparation.ipynb`.

In [2]:
import numpy as np
import json

from pathlib import Path
from scipy.spatial.distance import cdist
from time import perf_counter
from tqdm.auto import tqdm

DATA_DIR = Path("wikipedia_nn_data")
K = 10  # number of nearest neighbours to return

# Load embeddings
X = np.load(DATA_DIR / "embeddings.npy")
Q = np.load(DATA_DIR / "queries.npy")

# Load original sentences
with open(DATA_DIR / "sentences.json") as f:
    database_sentences = json.load(f)

with open(DATA_DIR / "query_sentences.json") as f:
    query_sentences = json.load(f)

print("Database shape:", X.shape)
print("Query shape:", Q.shape)

Database shape: (50000, 384)
Query shape: (500, 384)


## Brute-force search

For a query $q$ and database $X$ with $n$ vectors in $d$ dimensions:

1. Compute the squared Euclidean distance from $q$ to all $n$ vectors. This costs $O(n \cdot d)$. Squared distance gives the same ranking as distance and avoids a square root.
2. Use `np.argpartition` to find the $k$ smallest distances in $O(n)$.
3. Sort only those $k$ results.

The cost is independent of the data distribution: every query always looks at every vector.

In [3]:
def brute_force_search(X, q, k=10):
    """Exact k-NN: compare q with every vector in X and return the indices
    of the k closest, sorted from nearest to farthest. Cost: O(n·d) per query."""

    # Compute squared Euclidean distances
    distances = cdist(
        q.reshape(1, -1),
        X,
        metric="sqeuclidean"
    ).ravel()

    # Select the k closest vectors in O(n) time
    # (argpartition avoids a full O(n log n) sort of all distances)
    indices = np.argpartition(distances, k - 1)[:k]

    # Sort selected neighbors by distance
    indices = indices[np.argsort(distances[indices])]

    return indices

## Example query

A quick check that the embeddings capture meaning: the query is about a song, and its nearest neighbours are also sentences about songs and lyrics.

In [4]:
# Qualitative check: show the 10 nearest neighbours of the first query.
i = 0

neighbors = brute_force_search(X, Q[i], K)

print("Query:")
print(query_sentences[i])

print("\nTop 10 nearest neighbors:")

for rank, idx in enumerate(neighbors, 1):
    print(f"{rank}. {database_sentences[idx]}")

Query:
It samples the Scott McKenzie song, "San Francisco (Be Sure to Wear Flowers in Your Hair)", and "California Dreamin'" by The Mamas & the Papas.

Top 10 nearest neighbors:
1. The song's lyrics are an ode to the city of San Francisco in Northern California.
2. In addition to Fall's score, it featuring songs by Jerome Kern and others, with additional lyrics Adrian Ross and others.
3. The song's beat contains samples from "You Make the Sun Shine" by The Temprees and "What It Is" by The Temptations.
4. The song plays during the opening and end credits.
5. The song is all about wet dreams filled with sexual innuendos and pop culture references.
6. It is based on the lyrics from "A Dagger Through the Heart of St. Angeles".
7. It is a sentimental song in the tradition of "Don't Sit Under the Apple Tree" or "I'll Be Seeing You".
8. The song was used in a Rimmel make up advert.
9. The song was written by Stefani and Williams and produced by The Neptunes.
10. The song has been featured in 

## Ground truth

Brute force is exact, so its results are by definition the correct answers. We store them and use them to measure the **recall@10** of the other methods:

$$\text{recall@10} = \frac{|\text{returned neighbours} \cap \text{true neighbours}|}{10},$$

averaged over all queries.

In [5]:
# Ground truth: the exact top-10 neighbours of every query. These are the
# reference answers used to compute recall for every other method.
ground_truth = np.array([
    brute_force_search(X, q, K)
    for q in tqdm(Q, desc="Computing ground truth")
])

print("Ground truth shape:", ground_truth.shape)

Computing ground truth:   0%|          | 0/500 [00:00<?, ?it/s]

Ground truth shape: (500, 10)


In [6]:
# Save the ground truth for the other notebooks.
np.save(DATA_DIR / "ground_truth.npy", ground_truth)

print("Ground truth saved successfully.")

Ground truth saved successfully.


## Query time

We measure the average time per query over all 500 queries. Each measurement is repeated 3 times after a warm-up query, and we report the median to reduce noise from other processes on the machine.

The first cell times the search by hand; the second uses `evaluate_search` from `evaluation.py`, which is used for all methods so that they are measured the same way. The two numbers differ by a few milliseconds because of normal run-to-run variation.

In [7]:
# Manual timing of brute force: average time per query, median over 3 runs.
# Kept as a cross-check of the shared evaluate_search helper used below.
REPEATS = 3
times = []

# Warm-up
brute_force_search(X, Q[0], K)

for _ in range(REPEATS):

    start = perf_counter()

    for q in Q:
        brute_force_search(X, q, K)

    elapsed = perf_counter() - start
    times.append(elapsed / len(Q))

brute_query_time = np.median(times)

print(f"Query time: {brute_query_time * 1000:.3f} ms")

Query time: 31.170 ms


In [8]:
# The same measurement using the shared evaluation helper (recall + query time).
from evaluation import evaluate_search

brute_results = evaluate_search(
    search_fn=lambda q, k: brute_force_search(X, q, k),
    Q=Q,
    ground_truth=ground_truth,
    k=10,
    repeats=3
)

print(brute_results)

{'recall': 1.0, 'query_time_ms': 34.07158474996686}


## Save the baseline

In [9]:
# Store the baseline so the KD-tree notebook can compare against it.
# Brute force has no index, so its build time is 0.
import pandas as pd

baseline_results = pd.DataFrame([
    {
        "algorithm": "Brute force",
        "n": X.shape[0],
        "dimension": X.shape[1],
        "k": K,
        "recall_at_10": brute_results["recall"],
        "query_time_ms": brute_results["query_time_ms"],
        "build_time_s": 0.0
    }
])

baseline_results.to_csv(
    DATA_DIR / "baseline_results.csv",
    index=False
)

baseline_results

,algorithm,n,dimension,k,recall_at_10,query_time_ms,build_time_s
0,Brute force,50000,384,10,1.0,34.071585,0.0


## Summary

* Brute force takes about **34 ms per query** on 50,000 vectors in 384 dimensions.
* Its recall@10 is **1.0** by definition, since it is the ground truth.
* The ground truth is saved to `ground_truth.npy` and the baseline to `baseline_results.csv`.

The next notebook tests whether a KD-tree can answer the same exact queries faster.